In [ ]:
import os
import numpy as np
import pandas as pd

import pickle
from pathlib import Path

import os
import subprocess

In [ ]:
#just need the vaccine table from data for this exp, conditions table in data of other exp. 

In [ ]:
#vaccine

# This query represents dataset "all_vax_dataset" for domain "drug" and was generated for All of Us Controlled Tier Dataset v8




dataset_99852512_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (21601333, 21601349, 21601361, 21601366, 36118949, 37003432, 37498625, 40164828, 40166605, 40213140, 40213141, 40213142, 40213143, 40213144, 40213145, 40213146, 40213147, 40213148, 40213149, 40213150, 40213151, 40213152, 40213153, 40213154, 40213155, 40213156, 40213157, 40213158, 40213159, 40213160, 40213168, 40213170, 40213183, 40213184, 40213185, 40213186, 40213187, 40213188, 40213189, 40213190, 40213192, 40213193, 40213203, 40213223, 40213224, 40213251, 40213260, 40213284, 40213285, 40213286, 40213288, 40213289, 40213290, 40213293, 40213296, 40213298, 40213299, 40213300, 40213301, 40213302, 40213303, 40213304, 40213305, 40213306, 40213307, 40213308, 40213311, 40213317, 40213319, 40213320, 40213321, 40213322, 40213326, 40213327, 40220901, 40225028, 40225038, 42800027, 42873956, 42873961, 42899116, 42903441, 42903442, 42903942, 43012953, 43531944, 43532049, 43532418, 44112041, 44814322, 45776076, 45892474, 45892475, 45892476, 45892477,
 45892478, 46275993, 46275996, 46275999, 507832, 523212, 523283, 523365, 523367, 528323, 529076, 529112, 529114, 529116, 529660, 529713, 532272, 702664, 702665, 702671, 702672, 702673, 702680, 702681, 706103, 706104, 706105, 706109, 716023, 716024, 716025, 724893, 724894, 724896, 724898, 724899, 724902, 724904, 724905, 739902, 779890, 780152, 792777)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))) d_exposure 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
                ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
                ON d_exposure.drug_type_concept_id = d_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
                ON d_exposure.route_concept_id = d_route.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
                ON v.visit_concept_id = d_visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
                ON d_exposure.drug_source_concept_id = d_source_concept.concept_id"""

dataset_99852512_drug_df = pandas.read_gbq(
    dataset_99852512_drug_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_99852512_drug_df.head(5)

In [ ]:
#vaccine

def get_vaccine_table(vaccine_ids):
# This query represents dataset "all_vax_dataset" for domain "drug" and was generated for All of Us Controlled Tier Dataset v8

    # allow passing either a single int or a list/tuple of ints
    if not isinstance(vaccine_ids, (list, tuple)):
        vaccine_ids = [vaccine_ids]
    ids_sql = "(" + ",".join(str(i) for i in vaccine_ids) + ")"


    dataset_99852512_drug_sql = f"""
        SELECT
            d_exposure.person_id,
            d_exposure.drug_concept_id,
            d_standard_concept.concept_name as standard_concept_name,
            d_standard_concept.concept_code as standard_concept_code,
            d_standard_concept.vocabulary_id as standard_vocabulary,
            d_exposure.drug_exposure_start_datetime,
            d_exposure.drug_exposure_end_datetime,
            d_exposure.verbatim_end_date,
            d_exposure.drug_type_concept_id,
            d_type.concept_name as drug_type_concept_name,
            d_exposure.stop_reason,
            d_exposure.refills,
            d_exposure.quantity,
            d_exposure.days_supply,
            d_exposure.sig,
            d_exposure.route_concept_id,
            d_route.concept_name as route_concept_name,
            d_exposure.lot_number,
            d_exposure.visit_occurrence_id,
            d_visit.concept_name as visit_occurrence_concept_name,
            d_exposure.drug_source_value,
            d_exposure.drug_source_concept_id,
            d_source_concept.concept_name as source_concept_name,
            d_source_concept.concept_code as source_concept_code,
            d_source_concept.vocabulary_id as source_vocabulary,
            d_exposure.route_source_value,
            d_exposure.dose_unit_source_value 
        FROM
            ( SELECT
                * 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
            WHERE
                (
                    drug_concept_id IN (SELECT
                        DISTINCT ca.descendant_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                    JOIN
                        (SELECT
                            DISTINCT c.concept_id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id             
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                            WHERE
                                concept_id IN {ids_sql}              
                                AND full_text LIKE '%_rank1]%'       ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) b 
                            ON (ca.ancestor_id = b.concept_id)))) d_exposure 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
                    ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
                    ON d_exposure.drug_type_concept_id = d_type.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
                    ON d_exposure.route_concept_id = d_route.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                    ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
                    ON v.visit_concept_id = d_visit.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
                    ON d_exposure.drug_source_concept_id = d_source_concept.concept_id
            """


    dataset_99852512_drug_df = pd.read_gbq(
        dataset_99852512_drug_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook")
    
    return dataset_99852512_drug_df


In [ ]:
vax_df = get_vaccine_table(vaccine_ids)